# 09 — Multi-seed fine-tuning

Replicates the fine-tuning protocol of notebook 03 over ten seeds for the two strongest
encoders and five for the other two, changing nothing but the seed, and saves the weights
that notebooks 07 and 08 load.

Every model × seed unit is written to Drive as it finishes and skipped on re-run, so a
disconnected session costs one unit rather than the run.

- Input: `data/cleaned/*.json`
- Output: `results_r2_seeds/per_seed_scores_extended.csv`, `table_main_extended_seeds.csv`,
  `table_seed_wilcoxon.csv`, `weights/*.weights.h5`
- Runtime: ~6 h on an A100 for all 30 runs, resumable
- Reported in: paper Section VI-F

## 1. Config

In [ ]:
# CONFIG
REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2_seeds'

BACKEND = 'jax'     # 'jax' reproduces notebook 03 / the published results. 'tensorflow' also works;
                    # Keras 3 weight files are portable between backends either way.

# Protocol -- IDENTICAL to notebook 03. Do not change these.
SEQ_LEN          = 256
EPOCHS           = 5
EFFECTIVE_BATCH  = 16      # held constant by gradient accumulation during OOM backoff
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
PATIENCE         = 2
GLOBAL_SEED      = 42
USE_CLASS_WEIGHTS = True   # False = the optional no-weighting ablation for Minor comment 3

# 10 seeds where the ranking is disputed, 5 where it is not. Seeds 42, 1, 2 run
# first so that partial progress stays comparable with the three-seed results.
SEEDS_PER_MODEL = {
    'mDeBERTa-v3 (base)': [42, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'XLM-R (base)':       [42, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'mBERT (cased)':      [42, 1, 2, 3, 4],
    'DistilmBERT':        [42, 1, 2, 3, 4],
}

KNLP_MODELS = {
    'mDeBERTa-v3 (base)': ('deberta_v3',  'deberta_v3_base_multi'),
    'XLM-R (base)':       ('xlm_roberta', 'xlm_roberta_base_multi'),
    'mBERT (cased)':      ('bert',        'bert_base_multi'),
    'DistilmBERT':        ('distil_bert', 'distil_bert_base_multi'),
}

# Weights notebooks 07 and 08 will load. ~1.1 GB each.
# If Drive is tight, set this to {} -- but then 07 and 08 have nothing to load.
SAVE_WEIGHTS_FOR = {'mDeBERTa-v3 (base)': [42, 1, 2]}

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'keras>=3.3', 'keras-hub', 'scikit-learn', 'scipy', 'pandas'], check=False)

from google.colab import drive
drive.mount('/content/drive')

# Auto-load the shared paths written by notebook 00_setup_and_check.
try:
    import json as _json
    from pathlib import Path as _Path
    _p = _Path('/content/drive/MyDrive/r2_config.json')
    if _p.exists():
        _cfg = _json.load(open(_p))
        REPO_DIR = _cfg['REPO_DIR']
        RESULTS_DIR = _cfg.get('RESULTS_SEEDS_DIR', RESULTS_DIR)
        print('Paths loaded from r2_config.json')
    else:
        print('r2_config.json not found -- using the paths above. Run notebook 00 to generate it.')
except Exception as _e:
    print('Could not load r2_config.json (%s) -- using the paths above.' % type(_e).__name__)
print('REPO_DIR   =', REPO_DIR)
print('RESULTS_DIR=', RESULTS_DIR)


## 2. Imports and environment

In [ ]:
import os, json, gc, time, random, warnings, itertools
# Backend must be chosen BEFORE keras is imported.
os.environ['KERAS_BACKEND'] = BACKEND
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if BACKEND == 'jax':
    # JAX preallocates ~75% of VRAM by default, which starves the TF ops used
    # for tokenisation and makes OOM look like a model-size problem.
    os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
import keras
try:
    import keras_nlp as KNLP
except ImportError:
    import keras_hub as KNLP
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

# GPU ownership.
#
_gpus = tf.config.list_physical_devices('GPU')
if BACKEND == 'jax' and _gpus:
    try:
        tf.config.set_visible_devices([], 'GPU')
        print('TensorFlow: GPU hidden -- JAX owns the device (tokenisation stays on CPU)')
    except Exception as e:
        print('WARNING: could not hide the GPU from TensorFlow:', e)
        print('Restart the runtime and run this cell before any other TF op.')
else:
    for _d in _gpus:
        try:
            tf.config.experimental.set_memory_growth(_d, True)
        except Exception:
            pass

print(f'TF {tf.__version__} | Keras {keras.__version__} | backend {keras.backend.backend()}')
print('physical GPU:', [d.name for d in _gpus] or 'NONE')

# What card did Colab actually give us, and how much of it is free right now?
try:
    import subprocess as _sp
    _o = _sp.run(['nvidia-smi',
                  '--query-gpu=name,memory.total,memory.used,memory.free',
                  '--format=csv,noheader'], capture_output=True, text=True)
    if _o.returncode == 0 and _o.stdout.strip():
        print('nvidia-smi:', _o.stdout.strip())
        _free = int(_o.stdout.split(',')[-1].strip().split()[0])
        if _free < 9000:
            print(f'WARNING: only ~{_free} MiB free. mDeBERTa-v3 base full fine-tuning needs')
            print('roughly 9-10 GB (278M params x 4 copies for AdamW, plus activations).')
            print('If a previous cell is still holding memory, restart the runtime.')
except Exception:
    pass

if _gpus:
    keras.mixed_precision.set_global_policy('mixed_float16')
    print('mixed precision: mixed_float16')

ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}
LABELS = [0, 1, 2]

_REG = {
    'xlm_roberta': ('XLMRobertaClassifier', 'XLMRobertaPreprocessor'),
    'deberta_v3':  ('DebertaV3Classifier',  'DebertaV3Preprocessor'),
    'bert':        ('BertClassifier',       'BertPreprocessor'),
    'distil_bert': ('DistilBertClassifier', 'DistilBertPreprocessor'),
}

def slug(s):
    return s.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')

def load_split(name):
    d = pd.DataFrame(json.load(open(REPO / f'data/cleaned/{name}_cleaned.json', encoding='utf-8')))
    d['text'] = d['text'].astype(str)
    d['label'] = d['label'].astype(int)
    return d

REPO = Path(REPO_DIR); OUT = Path(RESULTS_DIR)
(OUT / 'probs').mkdir(parents=True, exist_ok=True)
(OUT / 'weights').mkdir(parents=True, exist_ok=True)

random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)
keras.utils.set_random_seed(GLOBAL_SEED)

train_df, dev_df, test_df = load_split('train'), load_split('dev'), load_split('test')
y_train = train_df['label'].to_numpy('int32')
y_dev   = dev_df['label'].to_numpy('int32')
y_test  = test_df['label'].to_numpy('int32')
print('Splits:', len(train_df), len(dev_df), len(test_df))

if USE_CLASS_WEIGHTS:
    cw = compute_class_weight('balanced', classes=np.array(LABELS), y=y_train)
    class_weight = {int(k): float(v) for k, v in zip(LABELS, cw)}
else:
    class_weight = None
print('class weights:', None if class_weight is None
      else {ID2LABEL[k]: round(v, 3) for k, v in class_weight.items()})

# --- the exact filenames notebooks 07 and 08 will look for ---
print('\nThis notebook will write these weight files:')
for base, seeds in SAVE_WEIGHTS_FOR.items():
    for sd in seeds:
        print(f'  {OUT / "weights" / f"{slug(base)}__seed{sd}.weights.h5"}')
print('Notebooks 07 and 08 expect exactly these names. If you change CLF_MODEL there, '
      'the slug must still match.')


## 3. Load data

In [ ]:
# Pre-tokenisation.
#
_PREPROC = {}

def get_preprocessor(family, preset, force_new=False):
    key = (family, preset)
    if key not in _PREPROC or force_new:
        _PREPROC[key] = getattr(KNLP.models, _REG[family][1]).from_preset(
            preset, sequence_length=SEQ_LEN)
    return _PREPROC[key]

def encode(texts, family, preset, chunk=256, name=''):
    texts = [str(t) for t in texts]
    parts = None
    for attempt in (0, 1):
        try:
            pre = get_preprocessor(family, preset, force_new=bool(attempt))
            parts = []
            with tf.device('/CPU:0'):
                for i in range(0, len(texts), chunk):
                    o = pre(tf.constant(texts[i:i + chunk], dtype=tf.string))
                    parts.append({k: np.asarray(v) for k, v in dict(o).items()})
            break
        except tf.errors.NotFoundError:
            if attempt:
                raise
            print('  SentencePiece resource invalidated -- rebuilding preprocessor, retrying')
    enc = {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}
    if name:
        print(f'  encoded {name}: {len(texts)} docs -> {tuple(next(iter(enc.values())).shape)}')
    return enc

def build_clf(family, preset, num_classes=3):
    # preprocessor=None -> the model takes token ids, not raw strings.
    try:
        return getattr(KNLP.models, _REG[family][0]).from_preset(
            preset, num_classes=num_classes, preprocessor=None)
    except Exception as e:
        raise RuntimeError(
            f'Could not load preset {preset!r} ({type(e).__name__}: {e}). '
            'Presets download from Kaggle -- check network access, and set '
            'KAGGLE_USERNAME / KAGGLE_KEY if prompted.') from e

def make_ds(X, y=None, batch=16, shuffle_seed=None):
    n = len(next(iter(X.values())))
    ds = tf.data.Dataset.from_tensor_slices((X, y) if y is not None else X)
    if shuffle_seed is not None:
        ds = ds.shuffle(n, seed=shuffle_seed, reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)


## 4. Classifier helpers (tokenise, build, train with OOM backoff, predict)

In [ ]:
# Memory management.
#
MICRO_BATCHES_PER_MODEL = {
    'mDeBERTa-v3 (base)': [16, 8],
    'XLM-R (base)':       [16, 8],
    'mBERT (cased)':      [16, 8],
    'DistilmBERT':        [16, 8],
}
DEFAULT_MICRO_BATCHES = [16, 8, 4]

# A run at micro-batch 16 uses no accumulation at all; anything smaller does.
# The notebook reports which was used per seed so you can state it in the paper.

def gpu_free_mib():
    try:
        import subprocess
        o = subprocess.run(['nvidia-smi', '--query-gpu=memory.free',
                            '--format=csv,noheader,nounits'],
                           capture_output=True, text=True)
        return int(o.stdout.strip().split('\n')[0]) if o.returncode == 0 else None
    except Exception:
        return None

def hard_reset(*objs):
    # Drop references, clear the Keras session, and clear JAX's compilation
    # caches. Without the JAX step, compiled executables keep device buffers
    # alive across seeds.
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    try:
        keras.backend.clear_session()
    except Exception:
        pass
    if BACKEND == 'jax':
        try:
            import jax
            jax.clear_caches()
        except Exception:
            pass
    gc.collect()

def is_oom(msg):
    m = str(msg)
    return ('RESOURCE_EXHAUSTED' in m or 'ResourceExhausted' in m
            or 'out of memory' in m.lower() or 'Out of memory' in m)

def compile_clf(clf, micro):
    assert EFFECTIVE_BATCH % micro == 0, 'EFFECTIVE_BATCH must divide by the micro-batch'
    accum = EFFECTIVE_BATCH // micro
    kw = {'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY}
    if accum > 1:
        try:
            opt = keras.optimizers.AdamW(gradient_accumulation_steps=accum, **kw)
        except TypeError:
            print(f'  NOTE: this Keras build has no gradient_accumulation_steps; running at '
                  f'true batch {micro} instead of an effective {EFFECTIVE_BATCH}. '
                  f'That is a protocol difference -- record it in the paper.')
            opt = keras.optimizers.AdamW(**kw)
    else:
        opt = keras.optimizers.AdamW(**kw)
    clf.compile(optimizer=opt,
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
    return accum

def fit_with_backoff(base, family, preset, seed, X_train, y_train, X_dev, y_dev, class_weight):
    micros = MICRO_BATCHES_PER_MODEL.get(base, DEFAULT_MICRO_BATCHES)
    last_msg = ''
    for micro in micros:
        clf = None
        try:
            hard_reset()
            keras.utils.set_random_seed(seed)
            clf = build_clf(family, preset)
            accum = compile_clf(clf, micro)
            free = gpu_free_mib()
            print(f'  training: micro-batch {micro} x accum {accum} = effective '
                  f'{EFFECTIVE_BATCH}' + (f'  |  {free} MiB free' if free else ''))
            hist = clf.fit(
                make_ds(X_train, y_train, batch=micro, shuffle_seed=seed),
                validation_data=make_ds(X_dev, y_dev, batch=micro),
                epochs=EPOCHS, class_weight=class_weight,
                callbacks=[keras.callbacks.EarlyStopping(
                    monitor='val_loss', patience=PATIENCE, restore_best_weights=True)],
                verbose=1)
            return clf, hist.history, micro
        except Exception as e:
            # Store a STRING, not the exception: keeping `e` keeps its traceback,
            # which keeps the frames, which keep this dead model on the GPU.
            last_msg = f'{type(e).__name__}: {e}'
            tb_free = is_oom(last_msg)
            clf = None
            hard_reset()
            if not tb_free:
                raise RuntimeError(last_msg)
            print(f'  OOM at micro-batch {micro} -- backing off '
                  f'({gpu_free_mib()} MiB free after cleanup)')
    raise MemoryError(
        f'Out of memory even at micro-batch {micros[-1]} for {base}.\n'
        f'  Free VRAM right now: {gpu_free_mib()} MiB.\n'
        f'  If an earlier seed succeeded and this one did not, memory is not being '
        f'released -- RESTART THE RUNTIME and re-run; finished seeds are skipped.\n'
        f'  Otherwise: Runtime > Change runtime type -> L4 or A100, or train the '
        f'smaller encoders first.\n'
        f'Last error: {last_msg}')

def predict_with_backoff(clf, X, start=None):
    for b in ([start] if start else []) + [64, 32, 16, 8]:
        try:
            return softmax(np.asarray(clf.predict(make_ds(X, batch=b), verbose=0)), axis=1)
        except Exception as e:
            if not is_oom(f'{type(e).__name__}: {e}'):
                raise
            print(f'  OOM predicting at batch {b} -- backing off')
            gc.collect()
    raise MemoryError('Out of memory predicting even at batch 8.')

def save_model_only_weights(clf, family, preset, path):
    # A compiled model's weight file carries AdamW's moment estimates too
    # (~4x the size). Copy to host, free the GPU, then save from a fresh
    # uncompiled model so the file holds parameters only.
    w = clf.get_weights()
    hard_reset(clf)
    tmp = build_clf(family, preset)
    tmp.set_weights(w)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp.save_weights(str(path))
    hard_reset(tmp)
    del w
    gc.collect()


## 5. Training loop (resumable; one unit per model x seed)

In [ ]:
completed, failed, log = [], [], []
t_start = time.time()

for base, seeds in SEEDS_PER_MODEL.items():
    family, preset = KNLP_MODELS[base]
    pending = [s for s in seeds
               if not (OUT / 'probs' / f'{slug(base)}__seed{s}.npy').exists()]
    if not pending:
        print(f'\n===== {base}: all {len(seeds)} seeds already done =====')
        completed += [(base, s) for s in seeds]
        continue

    print(f'\n===== {base} | {len(pending)} seed(s) to run =====')
    print('tokenising once for this family')
    X_train = encode(train_df['text'], family, preset, name='train')
    X_dev   = encode(dev_df['text'],   family, preset, name='dev')
    X_test  = encode(test_df['text'],  family, preset, name='test')

    for sd in seeds:
        pth = OUT / 'probs' / f'{slug(base)}__seed{sd}.npy'
        if pth.exists():
            print(f'[skip] {base} seed {sd}')
            completed.append((base, sd)); continue

        print(f'\n--- {base} | seed {sd} ---', flush=True)
        t0 = time.time()
        try:
            clf, hist, micro = fit_with_backoff(base, family, preset, sd, X_train, y_train,
                                                X_dev, y_dev, class_weight)
            probs = predict_with_backoff(clf, X_test)
        except Exception as e:
            print(f'  [FAIL] {type(e).__name__}: {e}')
            failed.append((base, sd, f'{type(e).__name__}: {e}'))
            hard_reset()
            continue

        np.save(pth, probs)
        yp = probs.argmax(1)
        mf1 = f1_score(y_test, yp, average='macro')
        mins = (time.time() - t0) / 60
        json.dump({'model': base, 'seed': sd, 'macro_f1': float(mf1),
                   'accuracy': float(accuracy_score(y_test, yp)),
                   'micro_batch': micro, 'effective_batch': EFFECTIVE_BATCH,
                   'minutes': round(mins, 1),
                   'history': {k: [float(x) for x in v] for k, v in hist.items()}},
                  open(OUT / 'probs' / f'{slug(base)}__seed{sd}.json', 'w'), indent=2)
        print(f'  macro-F1 = {mf1:.4f}  ({mins:.1f} min)')
        log.append({'model': base, 'seed': sd, 'macro_f1': mf1, 'minutes': mins})

        if sd in SAVE_WEIGHTS_FOR.get(base, []):
            wp = OUT / 'weights' / f'{slug(base)}__seed{sd}.weights.h5'
            try:
                save_model_only_weights(clf, family, preset, wp)
                clf = None                       # freed inside the helper
                print(f'  saved {wp.name}  ({wp.stat().st_size/1e6:.0f} MB) '
                      f'-- notebooks 07 and 08 will load this')
            except Exception as e:
                print(f'  [weights NOT saved] {type(e).__name__}: {e}')

        completed.append((base, sd))
        hard_reset(clf)
        _f = gpu_free_mib()
        if _f:
            print(f'  GPU free after cleanup: {_f} MiB')
            if _f < 6000:
                print('  WARNING: memory is not being released between seeds. If the next seed '
                      'OOMs, restart the runtime and re-run -- finished seeds are skipped.')

    del X_train, X_dev, X_test; gc.collect()

print(f'\n{len(completed)} runs complete, {len(failed)} failed, '
      f'{(time.time()-t_start)/60:.0f} min elapsed')
for f in failed:
    print('  ', f)


## 6. Confirm the weight handoff to notebooks 07 and 08

In [ ]:
# ---- confirm the handoff to notebooks 07 and 08 ----------------------
print('Weight files now on disk:')
found = sorted((OUT / 'weights').glob('*.weights.h5'))
for p in found:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.0f} MB)')
if not found:
    print('  NONE -- notebooks 07 and 08 will have nothing to load.')
    print('  Check SAVE_WEIGHTS_FOR above and that at least one of those seeds completed.')
else:
    print(f'\nSet WEIGHTS_DIR = {OUT / "weights"} in notebooks 07 and 08.')
    print('If you ran notebook 00, r2_config.json already points them here.')


## 7. Per-seed aggregation and ensembling

In [ ]:
seed_probs, seed_rows = {}, []
for base, seeds in SEEDS_PER_MODEL.items():
    arrs = []
    for sd in seeds:
        p = OUT / 'probs' / f'{slug(base)}__seed{sd}.npy'
        if not p.exists():
            continue
        pr = np.load(p); arrs.append(pr)
        yp = pr.argmax(1)
        seed_rows.append({'model': base, 'seed': sd,
                          'macro_f1': f1_score(y_test, yp, average='macro'),
                          'accuracy': accuracy_score(y_test, yp)})
    if arrs:
        seed_probs[base] = np.stack(arrs)

seeds_df = pd.DataFrame(seed_rows)
seeds_df.to_csv(OUT / 'per_seed_scores_extended.csv', index=False)

summary = []
for base, arr in seed_probs.items():
    s = seeds_df[seeds_df.model == base]['macro_f1']
    ens = arr.mean(0); yp = ens.argmax(1)
    per = f1_score(y_test, yp, average=None, labels=LABELS)
    summary.append({
        'model': base, 'n_seeds': len(arr),
        'macroF1_seed_mean': round(s.mean(), 4),
        'macroF1_seed_std': round(s.std(ddof=1), 4) if len(s) > 1 else 0.0,
        'macroF1_seed_min': round(s.min(), 4), 'macroF1_seed_max': round(s.max(), 4),
        'macro_f1_ens': round(f1_score(y_test, yp, average='macro'), 4),
        'accuracy_ens': round(accuracy_score(y_test, yp), 4),
        'weighted_f1_ens': round(f1_score(y_test, yp, average='weighted'), 4),
        'f1_Human': round(per[0], 4), 'f1_AI-Generated': round(per[1], 4),
        'f1_AI-Obfuscated': round(per[2], 4),
        'roc_auc_ovr': round(roc_auc_score(y_test, ens, multi_class='ovr', average='macro'), 4),
    })

summary_df = pd.DataFrame(summary).sort_values('macro_f1_ens', ascending=False)
summary_df.to_csv(OUT / 'table_main_extended_seeds.csv', index=False)
print(summary_df.to_string(index=False))

print('\nSANITY CHECK: the mDeBERTa-v3 ensemble should land near the published 0.917 macro-F1.')
print('If it is far off, the preset, the label mapping or the split is wrong -- stop and')
print('resolve that before any of these numbers reach the manuscript.')


## 8. Wilcoxon signed-rank tests on paired per-seed scores

In [ ]:
from scipy.stats import wilcoxon

def holm(pvals):
    idx = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(idx):
        run = max(run, (m - rank) * pvals[i])
        adj[i] = min(run, 1.0)
    return adj

rows = []
for a, b in itertools.combinations(list(seed_probs), 2):
    sa = seeds_df[seeds_df.model == a].set_index('seed')['macro_f1']
    sb = seeds_df[seeds_df.model == b].set_index('seed')['macro_f1']
    common = sorted(set(sa.index) & set(sb.index))
    if len(common) < 5:
        rows.append({'model_A': a, 'model_B': b, 'n_seeds': len(common),
                     'median_delta': np.nan, 'p_raw': np.nan,
                     'note': 'fewer than 5 paired seeds'})
        continue
    da, db = sa.loc[common].to_numpy(), sb.loc[common].to_numpy()
    try:
        _, p = wilcoxon(da, db)
    except ValueError:
        p = 1.0
    rows.append({'model_A': a, 'model_B': b, 'n_seeds': len(common),
                 'median_delta': round(float(np.median(da - db)), 4),
                 'p_raw': round(float(p), 4),
                 'note': f'min attainable two-sided p = {2**-(len(common)-1):.4f}'})

wil = pd.DataFrame(rows)
mask = ~pd.isna(wil['p_raw'])
adj = np.full(len(wil), np.nan)
if mask.sum():
    adj[mask.to_numpy()] = holm(np.array(wil.loc[mask, 'p_raw'], float))
wil['p_holm'] = np.round(adj, 4)
wil['significant'] = wil['p_holm'] < 0.05
wil.to_csv(OUT / 'table_seed_wilcoxon.csv', index=False)
print(wil.to_string(index=False))
print('\nThis is the test that answers the reviewer: the bootstrap varies test documents and')
print('McNemar varies nothing, but only this varies the random seed.')


## 9. Ensemble-size curve

In [ ]:
# Ensemble-size curve -- where does averaging stop helping?
rows = []
for base, arr in seed_probs.items():
    for k in range(1, len(arr) + 1):
        yp = arr[:k].mean(0).argmax(1)
        rows.append({'model': base, 'n_seeds_in_ensemble': k,
                     'macro_f1': round(f1_score(y_test, yp, average='macro'), 4)})
curve = pd.DataFrame(rows)
curve.to_csv(OUT / 'table_ensemble_size_curve.csv', index=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4.4))
for base, g in curve.groupby('model'):
    plt.plot(g['n_seeds_in_ensemble'], g['macro_f1'], marker='o', label=base)
plt.xlabel('seeds in probability ensemble'); plt.ylabel('test macro-F1')
plt.title('Ensemble size vs. macro-F1'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / 'fig_ensemble_size.png', dpi=200); plt.show()
